In [1]:
# --- 1. CONFIGURATION FOR CPU EXECUTION (Mandatory) ---
import tensorflow as tf
import os
import sys

# 1. Force TensorFlow to only see and use the CPU
tf.config.set_visible_devices([], 'GPU')

# 2. Disable all XLA/JIT optimizations (This directly addresses the "JIT compilation failed" error)
tf.config.optimizer.set_jit(False)

# Check to confirm GPU is not visible
gpus = tf.config.experimental.list_physical_devices('GPU')
if not gpus:
    print("✅ Successfully configured to run on CPU.")
else:
    print("❌ Configuration failed. Please ensure the kernel was restarted.")

# --- Now proceed with all other imports and code ---
from tensorflow.keras.optimizers import Adam 
# ... and the rest of your training script ...

❌ Configuration failed. Please ensure the kernel was restarted.


In [2]:
# 1. Install necessary libraries (run this cell once)
# !pip install pandas tensorflow scikit-learn

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# --- Configuration ---
CSV_FILE_PATH = "/home/chinmoy/Downloads/Training_data_for_novels.csv" # Ensure this path is correct
RANDOM_SEED = 42 

# 2. Load the data
df = pd.read_csv(CSV_FILE_PATH)

# Display the distribution to confirm classes
print("--- Loaded Target Distribution ---")
print(df['target'].value_counts())
print("-" * 30)

# 3. Define the final vocabulary and numerical map
FINAL_LABELS = ['NARRATOR', 'MAN1', 'WOMAN1', 'MAN2', 'WOMAN2']
label_to_id = {label: i for i, label in enumerate(FINAL_LABELS)}
NUM_CLASSES = len(FINAL_LABELS)

# Filter out any labels not in the FINAL_LABELS list (shouldn't happen if data is clean)
df = df[df['target'].isin(FINAL_LABELS)].reset_index(drop=True)

--- Loaded Target Distribution ---
target
NARRATOR    167
MAN1         40
MAN2         36
WOMAN1       35
WOMAN2       27
Name: count, dtype: int64
------------------------------


In [3]:
# --- Tokenization and Padding Parameters ---
MAX_WORDS = 20000  # Max vocabulary size
MAX_LEN = 100      # Max length of a sequence (Current Line + Context)

# 1. Initialize and Fit Tokenizer
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<unk>")
# Fit on the entire 'line' column (which contains the context sequences)
tokenizer.fit_on_texts(df['line'])

# 2. Convert text to numerical sequences
X_sequences = tokenizer.texts_to_sequences(df['line'])

# 3. Pad sequences
X_padded = pad_sequences(X_sequences, maxlen=MAX_LEN, padding='post', truncating='post')

# 4. Prepare Y labels (convert text labels to one-hot vectors)
Y_ids = np.array([label_to_id.get(label) for label in df['target']])
Y_one_hot = to_categorical(Y_ids, num_classes=NUM_CLASSES)

# 5. Split Data into Training and Testing Sets
# 80% for training the model, 20% for testing its performance
X_train, X_test, Y_train, Y_test = train_test_split(
    X_padded, 
    Y_one_hot, 
    test_size=0.2, 
    random_state=RANDOM_SEED,
    # Use stratify to ensure all classes (MAN1, WOMAN2, etc.) are represented proportionally
    stratify=Y_one_hot 
)

print(f"Total training samples: {X_train.shape[0]}")
print(f"Total testing samples: {X_test.shape[0]}")

Total training samples: 244
Total testing samples: 61


In [4]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

# --- Model Hyperparameters ---
EMBEDDING_DIM = 128
LSTM_UNITS = 64
DROPOUT_RATE = 0.3

# 1. Build the Model Architecture
model = Sequential([
    # Input Layer: Converts token IDs into dense numerical vectors
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    
    # Core RNN: Bidirectional LSTM
    # return_sequences=False because we are doing sequence-to-one (sequence -> 1 character label)
    Bidirectional(LSTM(LSTM_UNITS)),
    
    Dropout(DROPOUT_RATE),
    
    # Output Layer: Maps to the 5 classes (NARRATOR, MAN1, etc.) using softmax probability
    Dense(NUM_CLASSES, activation='softmax') 
])

custom_adam = Adam(clipvalue=1.0) 

# 2. Compile the Model using the custom optimizer
model.compile(
    optimizer=custom_adam,  # <--- Use the clipped optimizer here
    loss='categorical_crossentropy', 
    metrics=['accuracy']
)

print("Attempting training with Gradient Clipping (Adam(clipvalue=1.0))...")

# 3. Train the Model (Your existing fit call)
history = model.fit(
    X_train, Y_train,
    epochs=15, 
    batch_size=32,
    validation_data=(X_test, Y_test),
    verbose=1
)

# 2. Compile the Model
# model.compile(
#     optimizer='adam', 
#     loss='categorical_crossentropy', 
#     metrics=['accuracy']
# )

# print("\n--- Model Summary ---")
# model.summary()

# # 3. Train the Model
# print("\n--- Starting Training ---")
# history = model.fit(
#     X_train, Y_train,
#     epochs=15, # Start with 15 epochs; you can increase or decrease this
#     batch_size=32,
#     validation_data=(X_test, Y_test),
#     verbose=1
# )

Attempting training with Gradient Clipping (Adam(clipvalue=1.0))...
Epoch 1/15


/home/chinmoy/Final Year Project/demo_tts/venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2025-11-03 01:57:58.545321: I external/local_xla/xla/service/service.cc:163] XLA service 0x7e2f5c00c9f0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2025-11-03 01:57:58.545343: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Host, Default Version
2025-11-03 01:57:58.585093: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


2/8 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.2812 - loss: 1.5924

I0000 00:00:1762115279.231474   15999 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 246ms/step - accuracy: 0.4877 - loss: 1.4921 - val_accuracy: 0.5574 - val_loss: 1.3095
Epoch 2/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 74ms/step - accuracy: 0.5451 - loss: 1.3249 - val_accuracy: 0.5574 - val_loss: 1.2983
Epoch 3/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 75ms/step - accuracy: 0.5451 - loss: 1.2996 - val_accuracy: 0.5574 - val_loss: 1.2936
Epoch 4/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - accuracy: 0.5451 - loss: 1.2806 - val_accuracy: 0.5574 - val_loss: 1.2895
Epoch 5/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 75ms/step - accuracy: 0.5451 - loss: 1.2233 - val_accuracy: 0.5574 - val_loss: 1.2961
Epoch 6/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - accuracy: 0.5451 - loss: 1.1599 - val_accuracy: 0.5574 - val_loss: 1.3434
Epoch 7/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 75ms/step - accuracy: 0.5779 - loss: 1.0542 - val_accuracy: 0.5246 - val_loss: 1.2788
Epoch 8/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 78ms/step - accuracy: 0.6148 - loss: 0.8957 - val_accuracy: 0.5410 - val_loss: 1.4723
Epoch 9/15

In [14]:
import pdfplumber
import pandas as pd

def extract_and_structure_pdf(pdf_path):
    """Extracts text line-by-line from a PDF and cleans it."""
    text_data = []
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                # Using layout=True helps preserve block order for script/novel text
                text = page.extract_text(layout=True) 
                if text:
                    # Split into lines and process each line
                    lines = text.split('\n')
                    for line in lines:
                        # Basic cleaning: strip whitespace and check if not empty
                        clean_line = line.strip()
                        if clean_line:
                            text_data.append({
                                'page': page_num + 1,
                                'line_text': clean_line,
                                'original_line': line
                            })
    except FileNotFoundError:
        print(f"Error: PDF file not found at {pdf_path}. Please check the path.")
        return pd.DataFrame()

    return pd.DataFrame(text_data)

print("✅ 'extract_and_structure_pdf' function defined.")

✅ 'extract_and_structure_pdf' function defined.


In [16]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def automated_attribution_pipeline(pdf_path, model, tokenizer, label_to_id, max_len, context_size=2):
    """
    Processes a new PDF, line-by-line, and predicts the character label using the 
    trained RNN model based on the contextual flow.
    """
    
    # 1. Setup
    # Get the raw lines from the new PDF
    new_pdf_lines = extract_and_structure_pdf(pdf_path)['line_text'].tolist() 
    
    id_to_label = {v: k for k, v in label_to_id.items()}
    attributed_script = []
    
    # Initialize the context buffer with two generic NARRATOR starting lines
    # This maintains the correct sequence length (MAX_LEN) expectation for the model
    context_buffer = [
        f"[{id_to_label[0]}]: SCENE START", 
        f"[{id_to_label[0]}]: ACTION START"
    ] 

    print(f"Starting attribution for {len(new_pdf_lines)} lines...")
    
    # 2. Iterate and Predict
    for i, current_line_text in enumerate(new_pdf_lines):
        # Skip empty lines that might result from extraction
        if not current_line_text.strip():
            continue

        # --- A. Create Input Sequence (X) ---
        # The sequence is: [PREV_PREV_LABEL]: PREV_PREV_TEXT | [PREV_LABEL]: PREV_TEXT | CURRENT_TEXT
        context_string = " | ".join(context_buffer)
        input_sequence = f"{context_string} | {current_line_text}"
        
        # 3. Tokenize and Pad the sequence
        X_seq = tokenizer.texts_to_sequences([input_sequence])
        X_pad = pad_sequences(X_seq, maxlen=max_len, padding='post', truncating='post')
        
        # 4. Predict the character
        # Model predicts probabilities for the 5 classes (NARRATOR, MAN1, etc.)
        prediction_probabilities = model.predict(X_pad, verbose=0)
        predicted_id = np.argmax(prediction_probabilities)
        predicted_label = id_to_label[predicted_id]
        
        # 5. Store the Result
        attributed_script.append({
            'line_id': i + 1,
            'text': current_line_text,
            'predicted_character': predicted_label
        })
        
        # 6. Update the Context Buffer (The core of inference)
        # The model's *own* prediction is used as the next piece of context
        new_context_item = f"[{predicted_label}]: {current_line_text}"
        context_buffer.append(new_context_item)
        
        # Keep the buffer size limited to match CONTEXT_SIZE used during training
        if len(context_buffer) > context_size:
            context_buffer.pop(0) 

    return pd.DataFrame(attributed_script)

# ----------------------------------------------------------------------
# --- 3. Execution (Assuming the trained objects are in memory) ---
# ----------------------------------------------------------------------
NEW_PDF_PATH = "/home/chinmoy/Downloads/The-Invisible-Man.pdf" 

# Note: MAX_LEN, tokenizer, model, and label_to_id must be loaded from your training script.
df_attributed = automated_attribution_pipeline(
    pdf_path=NEW_PDF_PATH,
    model=model,
    tokenizer=tokenizer,
    label_to_id=label_to_id,
    max_len=MAX_LEN
)

print("\n--- Predicted Character Output Sample ---")
print(df_attributed.head(15))

Starting attribution for 4320 lines...

--- Predicted Character Output Sample ---
    line_id                                               text  \
0         1  The Project Gutenberg EBook of The Invisible M...   
1         2  This eBook is for the use of anyone anywhere a...   
2         3  almost no restrictions whatsoever. You may cop...   
3         4  re-use it under the terms of the Project Guten...   
4         5     with this eBook or online at www.gutenberg.net   
5         6                           Title: The Invisible Man   
6         7                                Author: H. G. Wells   
7         8        Release Date: October 7, 2004 [EBook #5230]   
8         9                        [Last updated: May 3, 2012]   
9        10                                  Language: English   
10       11  *** START OF THIS PROJECT GUTENBERG EBOOK THE ...   
11       12                             Produced by Andrew Sly   
12       13                         The    Invisible       M

In [18]:
print(df_attributed.iloc[300:400])

     line_id                                               text  \
300      301  "Yes," said Teddy. "By the week. Whatever he i...   
301      302  the week. And he's got a lot of luggage coming...   
302      303                it won't be stones in boxes, Hall."   
303      304  He told Hall how his aunt at Hastings had been...   
304      305  empty portmanteaux. Altogether he left Hall va...   
..       ...                                                ...   
395      396  Directly the first crate was, in accordance wi...   
396      397  parlour, the stranger flung himself upon it wi...   
397      398  began to unpack it, scattering the straw with ...   
398      399  carpet. And from it he began to produce bottle...   
399      400  powders, small and slender bottles containing ...   

    predicted_character  
300              WOMAN2  
301                MAN2  
302                MAN2  
303                MAN2  
304            NARRATOR  
..                  ...  
395          

In [20]:
print(df_attributed.iloc[1000:1010])

      line_id                                               text  \
1000     1001  "It's strange, perhaps, but it's not a crime. ...   
1001     1002                                     this fashion?"   
1002     1003  "Ah! that's a different matter," said Jaffers....   
1003     1004  see in this light, but I got a warrant and it'...   
1004     1005  invisibility,—it's burglary. There's a house b...   
1005     1006                                            "Well?"   
1006     1007               "And circumstances certainly point—"   
1007     1008      "Stuff and nonsense!" said the Invisible Man.   
1008     1009    "I hope so, sir; but I've got my instructions."   
1009     1010  "Well," said the stranger, "I'll come. I'll co...   

     predicted_character  
1000                MAN2  
1001            NARRATOR  
1002                MAN2  
1003            NARRATOR  
1004                MAN2  
1005                MAN2  
1006            NARRATOR  
1007                MAN1  
1008    